# SHAP Top-10 Feature Plots for Selected BBB Models

This notebook loads existing trained models and held-out validation splits. It does not retrain models.

Primary plots are SHAP summary/beeswarm plots because these show SHAP value on the x-axis and use a continuous color scale for feature values. Companion single-molecule SHAP waterfall plots are also generated for each model.

In [ ]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd / "brainroute_ml_validation"]
    for candidate in candidates:
        if (candidate / "brainroute_ml_validation" / "configs" / "validation_config.yaml").exists():
            return candidate
        if (candidate / "configs" / "validation_config.yaml").exists() and candidate.name == "brainroute_ml_validation":
            return candidate.parent
    raise FileNotFoundError("Could not locate repository root.")


REPO_ROOT = find_repo_root()
VALIDATION_ROOT = REPO_ROOT / "brainroute_ml_validation"
sys.path.insert(0, str(REPO_ROOT))

from brainroute_ml_validation.src.features import load_feature_view
from brainroute_ml_validation.src.modeling import selected_features_from_pipeline, split_data_for_feature_view
from brainroute_ml_validation.src.utils import load_config

CONFIG_PATH = VALIDATION_ROOT / "configs" / "validation_config.yaml"
FIGURE_DIR = VALIDATION_ROOT / "reports" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

cfg = load_config(CONFIG_PATH)
print(f"Repository root: {REPO_ROOT}")
print(f"SHAP version: {shap.__version__}")

## Model Choices

The duplicate-aware models use `duplicate_aware_seed5`, which was the strongest duplicate-aware split for both PaDEL + Morgan LightGBM and PaDEL + Morgan Extra Trees. The scaffold-CV XGBoost model uses `scaffold_cv_fold1`, which was the strongest scaffold-CV fold for PaDEL + Morgan + ChemBERTa XGBoost.

These choices can be changed in the `MODEL_RUNS` list below.

In [ ]:
RANDOM_SEED = 42
MAX_SHAP_ROWS = 500
MAX_DISPLAY = 10

MODEL_RUNS = [
    {
        "feature_view": "padel_morgan",
        "model": "lightgbm",
        "split": "duplicate_aware_seed5",
        "title": "PaDEL + Morgan / LightGBM / Duplicate-aware seed 5",
        "safe_name": "padel_morgan_lightgbm_duplicate_aware_seed5",
    },
    {
        "feature_view": "padel_morgan",
        "model": "extra_trees",
        "split": "duplicate_aware_seed5",
        "title": "PaDEL + Morgan / Extra Trees / Duplicate-aware seed 5",
        "safe_name": "padel_morgan_extra_trees_duplicate_aware_seed5",
    },
    {
        "feature_view": "padel_morgan_embeddings",
        "model": "xgboost",
        "split": "scaffold_cv_fold1",
        "title": "PaDEL + Morgan + ChemBERTa / XGBoost / Scaffold CV fold 1",
        "safe_name": "padel_morgan_embeddings_xgboost_scaffold_cv_fold1",
    },
]

## SHAP Helpers

In [ ]:
plt.rcParams.update(
    {
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "font.size": 10,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def class1_shap_values(raw_values):
    if isinstance(raw_values, list):
        return raw_values[1] if len(raw_values) > 1 else raw_values[0]
    arr = np.asarray(raw_values)
    if arr.ndim == 3 and arr.shape[-1] == 2:
        return arr[:, :, 1]
    return arr


def class1_expected_value(raw_expected_value):
    arr = np.asarray(raw_expected_value)
    if arr.ndim > 0 and arr.shape[0] > 1:
        return arr[1]
    if arr.ndim > 0:
        return arr[0]
    return float(arr)


def transformed_test_matrix(model, feature_view: str, split: str):
    X, index = load_feature_view(cfg, feature_view)
    _, test_df, _, X_test, _, y_test = split_data_for_feature_view(X, index, split, feature_view, cfg)
    X_trans = model[:-1].transform(X_test)
    feature_names = selected_features_from_pipeline(model)
    X_trans = pd.DataFrame(X_trans, columns=feature_names, index=test_df["molecule_id"].values)
    return test_df, X_trans, y_test


def sample_rows(X: pd.DataFrame, y, max_rows: int = MAX_SHAP_ROWS):
    if len(X) <= max_rows:
        return X, np.asarray(y)
    rng = np.random.default_rng(RANDOM_SEED)
    y = np.asarray(y)
    indices = []
    for label in sorted(np.unique(y)):
        label_idx = np.where(y == label)[0]
        n_label = max(1, round(max_rows * len(label_idx) / len(y)))
        n_label = min(n_label, len(label_idx))
        indices.extend(rng.choice(label_idx, size=n_label, replace=False).tolist())
    indices = np.array(sorted(indices[:max_rows]))
    return X.iloc[indices].copy(), y[indices]


def compute_shap_for_run(run: dict):
    model_path = VALIDATION_ROOT / "models" / f"{run['feature_view']}__{run['model']}__{run['split']}.joblib"
    model = joblib.load(model_path)
    test_df, X_trans, y_test = transformed_test_matrix(model, run["feature_view"], run["split"])
    X_sample, y_sample = sample_rows(X_trans, y_test)
    clf = model.named_steps["clf"]
    explainer = shap.TreeExplainer(clf)
    raw_values = explainer.shap_values(X_sample, check_additivity=False)
    values = class1_shap_values(raw_values)
    base_value = class1_expected_value(explainer.expected_value)
    mean_abs = np.abs(values).mean(axis=0)
    top_idx = np.argsort(mean_abs)[::-1][:MAX_DISPLAY]
    top_features = X_sample.columns[top_idx].tolist()
    return {
        "model": model,
        "model_path": model_path,
        "X_sample": X_sample,
        "y_sample": y_sample,
        "shap_values": values,
        "base_value": base_value,
        "top_idx": top_idx,
        "top_features": top_features,
        "top_mean_abs_shap": mean_abs[top_idx],
    }


def save_summary_plot(run: dict, result: dict):
    top_idx = result["top_idx"]
    X_top = result["X_sample"].iloc[:, top_idx]
    values_top = result["shap_values"][:, top_idx]
    plt.figure(figsize=(8.2, 6.2))
    shap.summary_plot(
        values_top,
        X_top,
        feature_names=X_top.columns.tolist(),
        max_display=MAX_DISPLAY,
        show=False,
        plot_size=None,
        color_bar=True,
    )
    plt.title(run["title"], pad=12)
    plt.xlabel("SHAP value for BBB+ prediction")
    output_path = FIGURE_DIR / f"shap_summary_top10__{run['safe_name']}.png"
    plt.tight_layout()
    plt.savefig(output_path, bbox_inches="tight", dpi=300)
    plt.close()
    return output_path


def save_waterfall_plot(run: dict, result: dict):
    values = result["shap_values"]
    row_index = int(np.argmax(np.abs(values).sum(axis=1)))
    explanation = shap.Explanation(
        values=values[row_index],
        base_values=result["base_value"],
        data=result["X_sample"].iloc[row_index].values,
        feature_names=result["X_sample"].columns.tolist(),
    )
    plt.figure(figsize=(8.2, 6.2))
    shap.plots.waterfall(explanation, max_display=MAX_DISPLAY, show=False)
    plt.title(run["title"], pad=12)
    output_path = FIGURE_DIR / f"shap_waterfall_top10__{run['safe_name']}.png"
    plt.tight_layout()
    plt.savefig(output_path, bbox_inches="tight", dpi=300)
    plt.close()
    return output_path


## Generate SHAP Plots

In [ ]:
outputs = []
importance_tables = []

for run in MODEL_RUNS:
    print(f"Processing {run['title']}")
    result = compute_shap_for_run(run)
    summary_path = save_summary_plot(run, result)
    waterfall_path = save_waterfall_plot(run, result)
    outputs.extend([summary_path, waterfall_path])
    importance_tables.append(
        pd.DataFrame(
            {
                "feature_view": run["feature_view"],
                "model": run["model"],
                "split": run["split"],
                "feature": result["top_features"],
                "mean_abs_shap": result["top_mean_abs_shap"],
            }
        )
    )

importance_summary = pd.concat(importance_tables, ignore_index=True)
importance_path = VALIDATION_ROOT / "reports" / "shap_top10_feature_summary_selected_models.csv"
importance_summary.to_csv(importance_path, index=False)

print("\nCreated figures:")
for path in outputs:
    print(path)
print(f"\nWrote top-10 SHAP feature summary: {importance_path}")
display(importance_summary)